# Metadata Enrichment
> Attach document IDs, chunk IDs, and custom metadata to ingested chunks.

Anchor provides ID generation utilities and a `MetadataEnricher` that lets
you inject arbitrary metadata into chunks during ingestion.

**Time:** ~5 minutes

## Setup

In [ ]:
from anchor.ingestion import DocumentIngester, MetadataEnricher
from anchor.ingestion.metadata import (
    generate_doc_id,
    generate_chunk_id,
    extract_chunk_metadata,
)
from anchor.ingestion.chunkers import RecursiveCharacterChunker
from anchor.models import SourceType

## Document & Chunk ID Generation
`generate_doc_id` creates a content-addressable hash for a document.
`generate_chunk_id` extends it with a chunk index for uniqueness.

In [ ]:
content = "Anchor is a framework for building context-aware AI applications."

doc_id = generate_doc_id(content)
print(f"Document ID: {doc_id}")

# Generate IDs for individual chunks
for i in range(3):
    chunk_id = generate_chunk_id(doc_id, index=i)
    print(f"  Chunk {i} ID: {chunk_id}")

## Content-Addressable Properties
The same content always produces the same document ID, making deduplication
straightforward.

In [ ]:
# Same content -> same ID
id_a = generate_doc_id("Hello, world!")
id_b = generate_doc_id("Hello, world!")
id_c = generate_doc_id("Hello, world?")

print(f"Same content:  {id_a == id_b}")  # True
print(f"Diff content:  {id_a == id_c}")  # False
print(f"\nID A: {id_a}")
print(f"ID B: {id_b}")
print(f"ID C: {id_c}")

## Extract Chunk Metadata
`extract_chunk_metadata` computes basic statistics about a chunk (word count,
character count, etc.).

In [ ]:
chunk_text = (
    "Anchor provides tools for memory management, retrieval-augmented "
    "generation, and intelligent context assembly."
)

meta = extract_chunk_metadata(chunk_text)

print("Chunk metadata:")
for key, value in meta.items():
    print(f"  {key}: {value}")

## Custom Enrichment Function
A `MetadataEnricher` wraps a user-defined function that receives chunks
and document-level metadata, and returns enriched metadata dicts.

In [ ]:
def enrich(chunks, doc_metadata):
    """Add word count and merge with document metadata."""
    return [
        {
            "chunk": c,
            "word_count": len(c.split()),
            **doc_metadata,
        }
        for c in chunks
    ]

enricher = MetadataEnricher(enrichment_fn=enrich)
print(f"Enricher ready: {type(enricher).__name__}")

## Apply Enrichment to Chunks

In [ ]:
# Chunk a sample document
chunker = RecursiveCharacterChunker(chunk_size=120, overlap=20)
sample_text = (
    "Anchor is a modular framework. It handles memory, retrieval, "
    "and context assembly. Developers can pick components they need. "
    "The ingestion module converts raw documents into context items."
)
chunks = chunker.chunk(sample_text)

# Enrich with document-level metadata
doc_meta = {"source": "overview.md", "version": "1.0"}
enriched = enricher.enrichment_fn(chunks, doc_meta)

print(f"Enriched {len(enriched)} chunks:\n")
for i, item in enumerate(enriched):
    print(f"  [{i}] word_count={item['word_count']}  "
          f"source={item['source']}  version={item['version']}")
    print(f"      {item['chunk'][:60]}...")
    print()

## Full Pipeline: IDs + Enrichment
Combine ID generation with enrichment for a complete metadata workflow.

In [ ]:
full_text = (
    "The context pipeline scores and filters items. Priority values "
    "determine ordering. Token budgets control how much context fits. "
    "Anchor supports retrieval results, conversation history, and tools."
)

doc_id = generate_doc_id(full_text)
chunks = chunker.chunk(full_text)

print(f"Document ID: {doc_id}")
print(f"Chunks: {len(chunks)}\n")

for i, chunk in enumerate(chunks):
    chunk_id = generate_chunk_id(doc_id, index=i)
    meta = extract_chunk_metadata(chunk)
    print(f"  Chunk {i}: id={chunk_id[:16]}...")
    for k, v in meta.items():
        print(f"    {k}: {v}")
    print()

## Key Takeaways
- `generate_doc_id()` creates deterministic, content-addressable document IDs
- `generate_chunk_id()` extends the doc ID with a chunk index
- `extract_chunk_metadata()` computes word count and other chunk statistics
- `MetadataEnricher` wraps a custom function to inject metadata during ingestion
- Combine IDs and enrichment for a complete provenance-tracking workflow

**Next:** [Parent-Child Chunks](05_parent_child_chunks.ipynb)